# LangChain Retriever Example: Olympic Running Events

This example demonstrates how to use the `RetrieverAdapter` from `aap_langchain` with a FAISS vector store to retrieve relevant documents about Olympic running events.

## Data Source
All data is scraped from the official Olympic website [olympics.com](https://olympics.com), covering:
- Sprint records (100m, 200m, 400m, hurdles)
- Marathon champions (1896-2024)
- Relay race history and records
- Athletics sport overview

## Workflow
1. Load markdown documents from the `data/` directory
2. Split text into chunks
3. Create embeddings using `all-MiniLM-L6-v2`
4. Build and save FAISS vector index
5. Query the retriever with natural language questions


In [17]:
# Import Required Libraries
from aap_langchain.retriever import RetrieverAdapter
from aap_core.types import AgentMessage

from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import CharacterTextSplitter


In [18]:
# Load Olympic Running Events Data
# Load all markdown files from the data directory
import os
import glob

data_dir = "data"
md_files = glob.glob(os.path.join(data_dir, "*.md"))

print(f"Found {len(md_files)} markdown files:")
for f in md_files:
    print(f"  - {f}")

# Load and combine all documents
documents = []
for md_file in md_files:
    loader = TextLoader(md_file, encoding="utf-8")
    docs = loader.load()
    for doc in docs:
        doc.metadata["source"] = os.path.basename(md_file)
    documents.extend(docs)

print(f"\nTotal documents loaded: {len(documents)}")


Found 4 markdown files:
  - data/athletics_overview.md
  - data/sprint_records.md
  - data/relay_history.md
  - data/marathon_champions.md

Total documents loaded: 4


In [19]:
# Split Text into Chunks
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
chunked_documents = text_splitter.split_documents(documents)

print(f"Total chunks after splitting: {len(chunked_documents)}")

# Create Embeddings and Vector Store
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Save FAISS index if it doesn't exist, otherwise load from it
faiss_index_path = "olympic_running_faiss"

if not os.path.exists(faiss_index_path):
    print("\nCreating FAISS index...")
    db = FAISS.from_documents(chunked_documents, embeddings)
    db.save_local(faiss_index_path)
    print(f"FAISS index saved to {faiss_index_path}/")
else:
    print("\nLoading existing FAISS index...")
    db = FAISS.load_local(faiss_index_path, embeddings, allow_dangerous_deserialization=True)
    total_vectors = db.index.ntotal
    print(f"FAISS index loaded with {total_vectors} vectors")

# Define the Retriever
retriever = db.as_retriever(search_kwargs={"k": 5})
print("\nRetriever configured with k=5")


Created a chunk of size 1426, which is longer than the specified 1000


Total chunks after splitting: 21

Loading existing FAISS index...
FAISS index loaded with 21 vectors

Retriever configured with k=5


In [20]:
# Test the Retriever with Olympic Running Events Queries
adapter = RetrieverAdapter(retriever=retriever)

# Query about Olympic sprint records
query = "Who holds the Olympic record for the men's 100m sprint?"
message = AgentMessage(query=query)
message = adapter(message)

print(f"Query: {query}")
print(f"\nRetrieved {len(message.context['data'])} documents:\n")
for i, doc in enumerate(message.context['data'], 1):
    print(f"--- Document {i} ---")
    print(doc[:500] + "..." if len(doc) > 500 else doc)
    print()


Query: Who holds the Olympic record for the men's 100m sprint?

Retrieved 5 documents:

--- Document 1 ---
Here's a list of all Olympic sprint records.

## 100m Olympic records

### Men's 100m

**Usain Bolt (Jamaica) - 9.63s at London 2012 Olympics (August 5, 2012)**

Jamaican sprinter Usain Bolt first took possession of the men's 100m Olympic record at the Beijing 2008 Olympics, clocking 9.69s to beat Canadian runner Donovan Bailey's 9.84s time set at the Atlanta 1996 Olympics.

At London 2012, Usain Bolt bettered his own record by 0.06 seconds to set a new Olympic record.

### Women's 100m

**Elain...

--- Document 2 ---
# Olympic Sprint Records (100m, 200m, 400m, Hurdles, Relays)
Source: https://olympics.com/en/news/olympic-records-sprint-athletics-relays-hurdles-100m-200m-400m
Scraped from: olympics.com (official Olympic site)

---

Olympic records in sprint events - Usain Bolt, Florence Griffith Joyner jewels in the crown

The list of records in sprint events like the 100m, 200m, 